<a href="https://colab.research.google.com/github/proinvestigadores/Abejas/blob/main/Smart_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Carga y auditoría de datos**

## **1.1. Setup y carga de datos**

In [6]:
# ---- 0) Configuración básica de entorno
import sys, os, re, math, json, numpy as np, pandas as pd
from datetime import datetime, timedelta

# Ajusta pandas para ver más columnas en prints
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

# ---- 1) Montar Google Drive y definir ruta del CSV
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Importar
folder_path = '/content/drive/MyDrive/Pigs/Smart_Manufacturing/Modelos/'
file_path = 'Bd_Procesada_Junio_2024.csv'
path = folder_path + file_path
df = pd.read_csv(path)
df.head()

Mounted at /content/drive


,Timestamp,MD1_RPM,MD2_RPM,MD3_RPM,MD4_RPM,MD5_RPM,MD1_P,MD2_P,MD3_P,MD4_P,MD5_P,MD1_APM,MD2_APM,MD3_APM,MD4_APM,MD5_APM,MD1_PORCENT,MD2_PORCENT,MD3_PORCENT,MD4_PORCENT,MD5_PORCENT,MD1_ZE_VP,MD2_ZE_VP,MD3_ZE_VP,MD4_ZE_VP,MD5_ZE_VP,MD1_FND,MD2_FND,MD3_FND,MD4_FND,MD5_FND,CBZ_Z1_SP,CBZ_Z1_VP,CBZ_Z2_SP,CBZ_Z2_VP,CBZ_Z3_SP,CBZ_Z3_VP,CBZ_Z4_SP,CBZ_Z4_VP,CBZ_Z5_SP,CBZ_Z5_VP,CBZ_Z6_SP,CBZ_Z6_VP,HAL1_VEL,HAL2_VEL,HAL3_VEL,HAL4_VEL,HAL5_VEL,HAL6_VEL
0,2024-03-19 16:30:11,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
1,2024-03-19 16:30:59,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
2,2024-03-19 16:31:58,10.0,3.0,3.0,3.0,10.0,3880.0,1250.0,225.0,795.0,928.0,11.570,12.530,1.7,1.7,18.365,0.88,1.32,1.32,1.39,1.23,35.2,31.5,34.6,34.0,34.6,273.9,255.0,271.8,264.5,269.4,249.0,251.9,249.0,249.0,250.0,249.7,250.0,253.8,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
3,2024-03-19 16:32:58,10.0,3.0,3.0,3.0,5.0,2928.0,1160.0,180.0,762.0,728.0,10.985,12.485,1.7,1.7,17.300,0.88,1.32,1.32,1.39,1.23,35.7,31.5,34.4,34.0,34.7,276.6,251.9,270.0,267.5,268.4,249.0,252.3,249.0,249.0,250.0,249.8,250.0,253.7,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
4,2024-03-19 16:33:58,10.0,3.0,3.0,3.0,5.0,2770.0,1137.0,158.0,735.0,548.0,10.960,12.420,1.7,1.7,16.125,0.88,1.32,1.32,1.39,1.23,35.8,31.4,34.4,34.0,34.2,279.4,249.0,266.8,270.5,267.5,249.0,252.4,249.0,249.2,250.0,250.0,250.0,253.4,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0


## **1.2. Funciones de auditoría de datos**

In [7]:
# =========================================
# BLOQUE 1 — SETUP + CARGA + AUDITORÍA
# Objetivo 3: Validación del modelo (paso 1)
# =========================================

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Estandariza nombres de columnas:
    - elimina espacios al inicio/fin
    - colapsa múltiples espacios internos
    - reemplaza espacios por '_'
    - asegura ASCII simple si hubiera tildes
    """
    def fix_col(c):
        # quita espacios externos
        c = c.strip()
        # colapsa espacios intermedios
        c = re.sub(r'\s+', ' ', c)
        # reemplaza espacios por underscore
        c = c.replace(' ', '_')
        # normaliza prefijos comunes tipo 'MD1' etc. sin tocar mayúsculas útiles
        return c
    df = df.rename(columns={c: fix_col(c) for c in df.columns})
    return df


def parse_timestamp(df: pd.DataFrame, ts_col_candidates=('Timestamp','timestamp','TIMESTAMP')) -> pd.DataFrame:
    """
    Busca la columna de tiempo, la parsea a datetime y ordena por tiempo.
    """
    ts_col = None
    for c in ts_col_candidates:
        if c in df.columns:
            ts_col = c
            break
    if ts_col is None:
        raise ValueError(f"No encontré columna de tiempo. Probé {ts_col_candidates} y no existen en df.columns={df.columns.tolist()[:10]}...")

    # Parse robusto (maneja 'YYYY-MM-DD HH:MM:SS' y variantes)
    df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
    if df[ts_col].isna().any():
        # Si hay fallos de parseo, explícitalos
        bad = df[df[ts_col].isna()]
        print("⚠️ Filas con Timestamp inválido:", len(bad))
        print(bad.head(3))
        # Puedes decidir aquí si las eliminas
        df = df.dropna(subset=[ts_col])

    df = df.sort_values(ts_col).reset_index(drop=True)
    return df, ts_col


def build_variable_families(columns):
    """
    Detecta familias de variables según prefijos/sufijos habituales del proceso:
    - RPM de módulos: MD*_RPM
    - Presiones: MD*_P (PSI)
    - % capa: MD*_PORCENT
    - Temperatura de fundido: MD*_FND
    - Temperaturas de cabezal: CBZ_Z{1..6}_SP y CBZ_Z{1..6}_VP
    - Velocidad haladores: HAL{1..6}_VEL
    Devuelve un dict con listas de columnas por familia.
    """
    fam = {
        "rpm_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_RPM", c)]),
        "presion_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_P", c)]),
        "porcentaje_capa": sorted([c for c in columns if re.fullmatch(r"MD\d+_PORCENT", c)]),
        "fundido_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_FND", c)]),
        "cabezal_SP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_SP", c)]),
        "cabezal_VP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_VP", c)]),
        "haladores_vel": sorted([c for c in columns if re.fullmatch(r"HAL\d+_VEL", c)]),
    }
    # Extras comunes en tu BD (si existen):
    extras = sorted([c for c in columns if c not in sum(fam.values(), []) and c not in ("Timestamp","timestamp","TIMESTAMP")])
    fam["extras"] = extras
    return fam


def time_gap_audit(ts: pd.Series) -> pd.DataFrame:
    """
    Calcula deltas consecutivos y los clasifica en bins relevantes para este proceso:
     - < 30s
     - 30s–59s
     - 59s–61s (ideal ≈ 60s)
     - 61s–120s
     - 2–5 min
     - 5–10 min
     - >10 min
    Devuelve tabla con conteos y %.
    """
    # diferencias en segundos
    dt = ts.sort_values().diff().dt.total_seconds().iloc[1:]  # omite la primera (NaN)
    bins = [-np.inf, 30, 59, 61, 120, 300, 600, np.inf]
    labels = ["<30s", "30–59s", "59–61s", "61–120s", "2–5min", "5–10min", ">10min"]
    cat = pd.cut(dt, bins=bins, labels=labels)
    counts = cat.value_counts().reindex(labels, fill_value=0)
    out = pd.DataFrame({
        "rango": counts.index,
        "conteo": counts.values,
        "porcentaje": np.round(100*counts.values / counts.sum(), 2)
    })
    out.loc["TOTAL"] = ["—", int(counts.sum()), 100.00]
    return out

# Carga del CSV
assert os.path.exists(path), f"No encuentro el CSV en: {path}"
df = pd.read_csv(path)

print("Forma (filas, columnas) antes de estandarizar:", df.shape)
df = standardize_columns(df)
df, TS_COL = parse_timestamp(df)

print("Columna de tiempo detectada:", TS_COL)
print("Rango temporal:", df[TS_COL].min(), "→", df[TS_COL].max())
print("Forma (filas, columnas) tras parseo/orden:", df.shape)
print("\nPrimeras filas:")
display(df.head(5))

# Diccionario de variables por familia
familias = build_variable_families(df.columns)
print("\nResumen de familias detectadas:")
for k, v in familias.items():
    print(f" - {k}: {len(v)} columnas")
    if len(v) > 0:
        print("   ", v)

# Auditoría de gaps temporales
gap_table = time_gap_audit(df[TS_COL])
print("\n=== Auditoría de gaps entre muestras ===")
display(gap_table)

# Chequeos rápidos de calidad de datos
# a) Porcentaje de valores nulos por columna (útil para decidir interpolaciones posteriores)
nulls = df.isna().mean().sort_values(ascending=False)
print("\nTop 10 columnas con mayor % de NaN:")
display((nulls*100).round(2).head(10).to_frame("%_NaN").T)

# b) Duplicados exactos por Timestamp (según milisegundo).
#    En el proceso, lo ideal es consolidar a muestreo por minuto en un bloque posterior.
dup_count = df.duplicated(subset=[TS_COL]).sum()
print(f"\nDuplicados exactos de {TS_COL}: {dup_count}")

# c) Vista de estadísticos básicos para familias principales (sirve para detectar sensores fuera de rango):
def describe_family(cols, titulo):
    if len(cols) == 0:
        print(f"\n[{titulo}] No se encontraron columnas.")
        return
    print(f"\n[{titulo}] Estadísticos rápidos")
    display(df[cols].describe().T)

describe_family(familias["rpm_modulos"], "RPM de módulos")
describe_family(familias["presion_modulos"], "Presión de módulos (PSI)")
describe_family(familias["fundido_modulos"], "Temperatura de fundido (°C)")
describe_family(familias["cabezal_SP"], "Cabezal SP (°C)")
describe_family(familias["cabezal_VP"], "Cabezal VP (°C)")
describe_family(familias["haladores_vel"], "Velocidad haladores (m/min aprox.)")

print("\n✅ BLOQUE 1 completado. Sube estas salidas al repo para continuar con el Bloque 2.")


Forma (filas, columnas) antes de estandarizar: (20260, 49)
Columna de tiempo detectada: Timestamp
Rango temporal: 2024-03-19 16:30:11 → 2024-04-02 16:50:06
Forma (filas, columnas) tras parseo/orden: (20260, 49)

Primeras filas:


,Timestamp,MD1_RPM,MD2_RPM,MD3_RPM,MD4_RPM,MD5_RPM,MD1_P,MD2_P,MD3_P,MD4_P,MD5_P,MD1_APM,MD2_APM,MD3_APM,MD4_APM,MD5_APM,MD1_PORCENT,MD2_PORCENT,MD3_PORCENT,MD4_PORCENT,MD5_PORCENT,MD1_ZE_VP,MD2_ZE_VP,MD3_ZE_VP,MD4_ZE_VP,MD5_ZE_VP,MD1_FND,MD2_FND,MD3_FND,MD4_FND,MD5_FND,CBZ_Z1_SP,CBZ_Z1_VP,CBZ_Z2_SP,CBZ_Z2_VP,CBZ_Z3_SP,CBZ_Z3_VP,CBZ_Z4_SP,CBZ_Z4_VP,CBZ_Z5_SP,CBZ_Z5_VP,CBZ_Z6_SP,CBZ_Z6_VP,HAL1_VEL,HAL2_VEL,HAL3_VEL,HAL4_VEL,HAL5_VEL,HAL6_VEL
0,2024-03-19 16:30:11,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
1,2024-03-19 16:30:59,20.0,3.0,3.0,3.0,10.0,4678.0,1322.0,245.0,805.0,945.0,12.100,12.495,1.7,1.7,18.380,0.88,1.32,1.32,1.39,1.23,35.8,31.5,34.9,34.0,34.2,273.1,258.1,269.6,263.9,270.4,249.0,251.4,249.0,249.0,250.0,249.7,250.0,253.6,251.0,250.8,251.0,251.0,40.4,42.2,34.0,94.5,95.3,73.0
2,2024-03-19 16:31:58,10.0,3.0,3.0,3.0,10.0,3880.0,1250.0,225.0,795.0,928.0,11.570,12.530,1.7,1.7,18.365,0.88,1.32,1.32,1.39,1.23,35.2,31.5,34.6,34.0,34.6,273.9,255.0,271.8,264.5,269.4,249.0,251.9,249.0,249.0,250.0,249.7,250.0,253.8,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
3,2024-03-19 16:32:58,10.0,3.0,3.0,3.0,5.0,2928.0,1160.0,180.0,762.0,728.0,10.985,12.485,1.7,1.7,17.300,0.88,1.32,1.32,1.39,1.23,35.7,31.5,34.4,34.0,34.7,276.6,251.9,270.0,267.5,268.4,249.0,252.3,249.0,249.0,250.0,249.8,250.0,253.7,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0
4,2024-03-19 16:33:58,10.0,3.0,3.0,3.0,5.0,2770.0,1137.0,158.0,735.0,548.0,10.960,12.420,1.7,1.7,16.125,0.88,1.32,1.32,1.39,1.23,35.8,31.4,34.4,34.0,34.2,279.4,249.0,266.8,270.5,267.5,249.0,252.4,249.0,249.2,250.0,250.0,250.0,253.4,251.0,250.8,251.0,250.9,40.4,42.2,34.0,94.5,95.3,73.0



Resumen de familias detectadas:
 - rpm_modulos: 5 columnas
    ['MD1_RPM', 'MD2_RPM', 'MD3_RPM', 'MD4_RPM', 'MD5_RPM']
 - presion_modulos: 5 columnas
    ['MD1_P', 'MD2_P', 'MD3_P', 'MD4_P', 'MD5_P']
 - porcentaje_capa: 5 columnas
    ['MD1_PORCENT', 'MD2_PORCENT', 'MD3_PORCENT', 'MD4_PORCENT', 'MD5_PORCENT']
 - fundido_modulos: 5 columnas
    ['MD1_FND', 'MD2_FND', 'MD3_FND', 'MD4_FND', 'MD5_FND']
 - cabezal_SP: 6 columnas
    ['CBZ_Z1_SP', 'CBZ_Z2_SP', 'CBZ_Z3_SP', 'CBZ_Z4_SP', 'CBZ_Z5_SP', 'CBZ_Z6_SP']
 - cabezal_VP: 6 columnas
    ['CBZ_Z1_VP', 'CBZ_Z2_VP', 'CBZ_Z3_VP', 'CBZ_Z4_VP', 'CBZ_Z5_VP', 'CBZ_Z6_VP']
 - haladores_vel: 6 columnas
    ['HAL1_VEL', 'HAL2_VEL', 'HAL3_VEL', 'HAL4_VEL', 'HAL5_VEL', 'HAL6_VEL']
 - extras: 10 columnas
    ['MD1_APM', 'MD1_ZE_VP', 'MD2_APM', 'MD2_ZE_VP', 'MD3_APM', 'MD3_ZE_VP', 'MD4_APM', 'MD4_ZE_VP', 'MD5_APM', 'MD5_ZE_VP']

=== Auditoría de gaps entre muestras ===


,rango,conteo,porcentaje
0,<30s,221,1.09
1,30–59s,3238,15.98
2,59–61s,14279,70.48
3,61–120s,2512,12.40
4,2–5min,9,0.04
5,5–10min,0,0.00
6,>10min,0,0.00
TOTAL,—,20259,100.00



Top 10 columnas con mayor % de NaN:


...


[Cabezal VP (°C)] Estadísticos rápidos


,count,mean,std,min,25%,50%,75%,max
CBZ_Z1_VP,20260.0,251.367270,0.766144,249.6,250.8,251.3,252.0,252.8
CBZ_Z2_VP,20260.0,248.725030,0.484118,244.6,248.3,249.0,249.0,249.9
CBZ_Z3_VP,20260.0,247.860464,0.757689,246.1,247.3,247.9,248.4,250.5
CBZ_Z4_VP,20260.0,251.406703,0.360667,250.7,251.1,251.4,251.7,253.9
CBZ_Z5_VP,20260.0,248.011841,0.251156,245.1,248.0,248.0,248.0,251.4
CBZ_Z6_VP,20260.0,248.012315,0.294222,243.8,247.9,248.0,248.1,252.5



[Velocidad haladores (m/min aprox.)] Estadísticos rápidos


,count,mean,std,min,25%,50%,75%,max
HAL1_VEL,20260.0,40.004980,1.046879,35.2,39.8,40.1,40.8,41.1
HAL2_VEL,20260.0,41.762665,1.092682,36.5,41.6,41.9,42.5,42.7
HAL3_VEL,20260.0,34.000000,0.000000,34.0,34.0,34.0,34.0,34.0
HAL4_VEL,20260.0,89.457295,2.645648,77.4,89.1,89.7,91.0,94.5
HAL5_VEL,20260.0,94.284970,2.450095,83.2,94.0,94.1,96.0,97.0
HAL6_VEL,20260.0,73.277853,1.930675,64.5,73.0,73.5,74.7,75.2



✅ BLOQUE 1 completado. Sube estas salidas al repo para continuar con el Bloque 2.


# **2. Filtro de outliers y etiquetado de estados**

## **2.1. Normalización temporal + filtro de outliers**

In [8]:
from IPython.display import display

# --- Validaciones mínimas ---
if 'df' not in globals() or 'TS_COL' not in globals():
    raise RuntimeError("No encuentro 'df' o 'TS_COL'. Ejecutar antes el Bloque 1.")

# --- Utilidades (definidas aquí para evitar dependencias del orden de celdas) ---
def build_variable_families(columns):
    import re
    fam = {
        "rpm_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_RPM", c)]),
        "presion_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_P", c)]),
        "porcentaje_capa": sorted([c for c in columns if re.fullmatch(r"MD\d+_PORCENT", c)]),
        "fundido_modulos": sorted([c for c in columns if re.fullmatch(r"MD\d+_FND", c)]),
        "cabezal_SP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_SP", c)]),
        "cabezal_VP": sorted([c for c in columns if re.fullmatch(r"CBZ_Z\d+_VP", c)]),
        "haladores_vel": sorted([c for c in columns if re.fullmatch(r"HAL\d+_VEL", c)]),
    }
    extras = sorted([c for c in columns if c not in sum(fam.values(), []) and c not in (TS_COL,)])
    fam["extras"] = extras
    return fam

def cap_outliers_df(df_num: pd.DataFrame, iqr_k: float = 3.0, p_low: float = 0.001, p_high: float = 0.999):
    """
    Aplica capping columna a columna usando dos criterios:
    - Regla de Tukey extendida: [Q1 - k*IQR, Q3 + k*IQR]
    - Percentiles (p_low, p_high) para robustecer límites
    Devuelve (df_capped, reporte_df) sin modificar el df de entrada.
    """
    capped = df_num.copy()
    rows = []
    for col in df_num.columns:
        s = df_num[col]
        if not np.issubdtype(s.dtype, np.number):
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if np.isfinite(iqr) and iqr > 0:
            low_iqr = q1 - iqr_k * iqr
            high_iqr = q3 + iqr_k * iqr
        else:
            # Si la columna es casi constante, usa solo percentiles
            low_iqr = s.min()
            high_iqr = s.max()
        low_p = s.quantile(p_low)
        high_p = s.quantile(p_high)
        low_cap = max(low_iqr, low_p)
        high_cap = min(high_iqr, high_p)

        s_clipped = s.clip(lower=low_cap, upper=high_cap)
        changed = (s_clipped != s) & s.notna()
        capped[col] = s_clipped

        rows.append({
            "columna": col,
            "n_total": int(s.notna().sum()),
            "capados": int(changed.sum()),
            "min_original": float(np.nanmin(s)),
            "p001": float(low_p),
            "Q1": float(q1),
            "Q3": float(q3),
            "p999": float(high_p),
            "max_original": float(np.nanmax(s)),
            "min_nuevo": float(np.nanmin(s_clipped)),
            "max_nuevo": float(np.nanmax(s_clipped))
        })
    report = pd.DataFrame(rows).sort_values("capados", ascending=False).reset_index(drop=True)
    return capped, report

# --- 1) Copia de trabajo y resampleo a 1 min ---
dfw = df.copy()
# Asegurar orden temporal por si acaso
dfw = dfw.sort_values(TS_COL).reset_index(drop=True)

# Selecciona solo numéricas para operaciones de promedio/interpolación
num_cols = dfw.select_dtypes(include='number').columns.tolist()

# Index temporal (DatetimeIndex) para resample; no se modifica df original
df_num = dfw.set_index(TS_COL)[num_cols]

# Resampleo a 1 min con promedio; mantiene únicamente columnas numéricas
df_num_1min = df_num.resample('1min').mean()

# Interpolación temporal (hasta 2 minutos seguidos de NaN).
# 'limit_direction="both"' rellena huecos al inicio/fin hasta el límite.
df_num_1min = df_num_1min.interpolate(method='time', limit=2, limit_direction='both')

print("Forma numérica tras resampleo a 1min:", df_num_1min.shape)

# --- 2) Capping de outliers (incluye el caso MD4_ZE_VP si existe) ---
df_capped, capping_report = cap_outliers_df(df_num_1min, iqr_k=3.0, p_low=0.001, p_high=0.999)

print("\n=== Bitácora de capping (top 10 por cantidad de valores capados) ===")
display(capping_report.head(10))

if "MD4_ZE_VP" in df_capped.columns:
    fila = capping_report[capping_report["columna"] == "MD4_ZE_VP"]
    if not fila.empty:
        print("\nResumen específico de MD4_ZE_VP:")
        display(fila)

# --- 3) Reconstrucción de df_1min (numérico limpio) y reporte rápido ---
df_1min = df_capped.copy()
df_1min.index.name = TS_COL  # etiqueta el índice para consistencia

print("\nEstadísticos rápidos post-capping (MD*_ZE_VP si existe):")
ze_cols = [c for c in df_1min.columns if c.endswith("_ZE_VP")]
if ze_cols:
    display(df_1min[ze_cols].describe().T)

# --- 4) Familias sobre el nuevo df_1min (para siguientes bloques) ---
familias_1min = build_variable_families(df_1min.columns)
print("\nFamilias detectadas en df_1min (numéricas):")
for k, v in familias_1min.items():
    if k != "extras":
        print(f" - {k}: {len(v)}")
print("Extras:", len(familias_1min["extras"]))

print("\n✅ BLOQUE 2A completado. Usar 'df_1min' para ingeniería de características y reglas de estado.")


Forma numérica tras resampleo a 1min: (20181, 48)

=== Bitácora de capping (top 10 por cantidad de valores capados) ===


,columna,n_total,capados,min_original,p001,Q1,Q3,p999,max_original,min_nuevo,max_nuevo
0,MD4_ZE_VP,20181,2295,33.7,33.80,33.90,34.00,106.194,6550.30,33.80,34.30
1,HAL4_VEL,20181,742,77.4,77.40,89.10,91.00,94.500,94.50,83.40,94.50
2,MD5_APM,20181,655,0.0,0.00,24.14,25.37,27.450,27.83,20.45,27.45
3,MD4_P,20181,564,60.0,70.63,3030.00,3550.00,4180.000,4500.00,1470.00,4180.00
4,MD2_P,20181,563,107.0,137.00,4244.00,5008.00,5650.280,6151.00,1952.00,5650.28
5,MD4_RPM,20181,543,0.0,0.00,27.00,33.00,41.000,42.00,9.00,41.00
6,MD4_APM,20181,542,0.0,0.00,15.90,19.40,24.100,24.70,5.40,24.10
7,MD2_RPM,20181,541,0.0,0.00,27.00,34.00,41.000,41.00,6.00,41.00
8,HAL5_VEL,20181,536,83.2,83.20,94.00,96.00,97.000,97.00,88.00,97.00
9,HAL2_VEL,20181,536,36.5,36.50,41.60,42.50,42.700,42.70,38.90,42.70



Resumen específico de MD4_ZE_VP:


,columna,n_total,capados,min_original,p001,Q1,Q3,p999,max_original,min_nuevo,max_nuevo
0,MD4_ZE_VP,20181,2295,33.7,33.8,33.9,34.0,106.194,6550.3,33.8,34.3



Estadísticos rápidos post-capping (MD*_ZE_VP si existe):


,count,mean,std,min,25%,50%,75%,max
MD1_ZE_VP,20181.0,35.627207,0.365160,35.00,35.3,35.600,35.900,37.364
MD2_ZE_VP,20181.0,31.957521,0.927722,29.65,31.3,31.750,32.800,36.582
MD3_ZE_VP,20181.0,34.758208,0.207461,34.30,34.6,34.775,34.900,35.800
MD4_ZE_VP,20181.0,34.002401,0.119312,33.80,33.9,34.000,34.000,34.300
MD5_ZE_VP,20181.0,34.515702,0.295526,34.00,34.3,34.500,34.725,35.382



Familias detectadas en df_1min (numéricas):
 - rpm_modulos: 5
 - presion_modulos: 5
 - porcentaje_capa: 5
 - fundido_modulos: 5
 - cabezal_SP: 6
 - cabezal_VP: 6
 - haladores_vel: 6
Extras: 10

✅ BLOQUE 2A completado. Usar 'df_1min' para ingeniería de características y reglas de estado.


## **2.2. Ventanas y reglas de estado**

1.   Este bloque parte desde *df_1min* (datetimelike a 1 min) y *familias_1min*.
2.   Construye _features_ por ventana de 5 minutos con métricas: nivel y variabilidad temporal de haladores, RPM, presión y desviación de cabezal.
3. Aprende umbrales automáticos a partir de cuantiles para distinguir: PARO, PRODUCCION, SETUP, DESPERDICIO.
4. Genera una línea de tiempo de estados y un reporte de conteos/porcentajes, más un diccionario con los umbrales utilizados (todo exportable a CSV/JSON).

In [9]:
# =========================================
# BLOQUE 2B — Ingeniería de características (5 min) + Reglas de estado
# Requisitos: df_1min (index de tiempo a 1 min), familias_1min, TS_COL
# =========================================
import numpy as np
import pandas as pd
from IPython.display import display

# Validaciones mínimas
if 'df_1min' not in globals():
    raise RuntimeError("No encuentro 'df_1min'. Ejecuta antes el Bloque 2A.")
if 'familias_1min' not in globals():
    raise RuntimeError("No encuentro 'familias_1min'. Ejecuta antes el Bloque 2A.")
if 'TS_COL' not in globals():
    raise RuntimeError("No encuentro 'TS_COL'. Ejecuta antes el Bloque 1/2A.")

# Funciones
def _safe_mean(df: pd.DataFrame, cols: list) -> pd.Series:
    """Media fila-a-fila sobre columnas numéricas existentes. Si no hay columnas, devuelve NaN."""
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(index=df.index, dtype='float64')
    return df[cols].mean(axis=1)

def _rolling_std(s: pd.Series, window='5min', min_periods=3) -> pd.Series:
    """Desviación estándar temporal con ventana tipo time-based (sin deprecaciones)."""
    return s.rolling(window=window, min_periods=min_periods).std()

def _rolling_mean(s: pd.Series, window='5min', min_periods=3) -> pd.Series:
    return s.rolling(window=window, min_periods=min_periods).mean()

def _rolling_mean_absdiff(df: pd.DataFrame, cols: list, window='5min', min_periods=3) -> pd.Series:
    """
    Inestabilidad por módulo/halador: |Δ| promedio (diferencia absoluta minuto a minuto)
    agregado entre columnas y promediado en el tiempo.
    """
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(index=df.index, dtype='float64')
    absdiff = df[cols].diff().abs().mean(axis=1)
    return absdiff.rolling(window=window, min_periods=min_periods).mean()

def _percent_zero_haladores(df: pd.DataFrame, hal_cols: list, window='5min', min_periods=3) -> pd.Series:
    """
    % de ceros en haladores dentro de la ventana (promedio temporal y entre haladores).
    Resultado en [0,1].
    """
    hal_cols = [c for c in hal_cols if c in df.columns]
    if not hal_cols:
        return pd.Series(index=df.index, dtype='float64')
    zeros = df[hal_cols].eq(0).astype('float64')
    # media temporal por halador y luego media entre haladores
    z_time = zeros.rolling(window=window, min_periods=min_periods).mean()
    return z_time.mean(axis=1)

def _robust_q(s: pd.Series, q: float, fallback=np.nan):
    """Cuantil robusto ignorando NaN; si no se puede, devuelve fallback."""
    try:
        v = s.quantile(q)
        if pd.isna(v):
            return fallback
        return float(v)
    except Exception:
        return fallback

# ---- Familias requeridas
HAL = familias_1min.get("haladores_vel", [])
RPM = familias_1min.get("rpm_modulos", [])
PRES = familias_1min.get("presion_modulos", [])
FND = familias_1min.get("fundido_modulos", [])
CBZ_SP = familias_1min.get("cabezal_SP", [])
CBZ_VP = familias_1min.get("cabezal_VP", [])

if not HAL:
    raise RuntimeError("No se encontraron columnas de haladores (HAL*_VEL). Son necesarias para el etiquetado.")
if not RPM:
    print("⚠️ No hay columnas MD*_RPM; se omitirá variabilidad de RPM en reglas.")
if not PRES:
    print("⚠️ No hay columnas MD*_P; se omitirá presión en reglas.")
if not CBZ_SP or not CBZ_VP:
    print("⚠️ Faltan CBZ_Z*_SP o CBZ_Z*_VP; se omitirá desviación de cabezal en reglas.")

# ---- Construcción de series base (nivel instantáneo por minuto)
hal_speed = _safe_mean(df_1min, HAL)                       # nivel de avance
rpm_level = _safe_mean(df_1min, RPM) if RPM else pd.Series(index=df_1min.index, dtype='float64')
pres_level = _safe_mean(df_1min, PRES) if PRES else pd.Series(index=df_1min.index, dtype='float64')

# Desviación de cabezal |VP - SP| promedio en zonas
if CBZ_SP and CBZ_VP:
    # Emparejar zonas por sufijo Z{n}
    # Construye lista de pares (VP, SP) por zona existente
    zones = []
    for sp in CBZ_SP:
        z = sp.replace("_SP","").split("_")[-1]  # "Z1"..."Z6"
        vp = sp.replace("_SP","_VP")
        if vp in df_1min.columns:
            zones.append((vp, sp))
    if zones:
        cbz_dev = pd.concat([ (df_1min[vp] - df_1min[sp]).abs() for (vp, sp) in zones ], axis=1).mean(axis=1)
    else:
        cbz_dev = pd.Series(index=df_1min.index, dtype='float64')
else:
    cbz_dev = pd.Series(index=df_1min.index, dtype='float64')

# ---- Features por ventana (5 min, paso 1 min)
WINDOW = '5min'
MINP = 3

features = pd.DataFrame(index=df_1min.index)
features["hal_speed_mean_5m"]   = _rolling_mean(hal_speed, WINDOW, MINP)
features["hal_speed_std_5m"]    = _rolling_std(hal_speed, WINDOW, MINP)
features["hal_pct_zero_5m"]     = _percent_zero_haladores(df_1min, HAL, WINDOW, MINP)

# Variabilidad temporal de RPM y Presión (si existen)
if not rpm_level.empty:
    features["rpm_level_mean_5m"] = _rolling_mean(rpm_level, WINDOW, MINP)
    features["rpm_var_t_5m"]      = _rolling_std(rpm_level, WINDOW, MINP)
    # Inestabilidad basada en diferencias minuto a minuto entre módulos
    features["rpm_absdiff_5m"]    = _rolling_mean_absdiff(df_1min, RPM, WINDOW, MINP)
else:
    features["rpm_level_mean_5m"] = np.nan
    features["rpm_var_t_5m"]      = np.nan
    features["rpm_absdiff_5m"]    = np.nan

if not pres_level.empty:
    features["pres_level_mean_5m"] = _rolling_mean(pres_level, WINDOW, MINP)
    features["pres_var_t_5m"]      = _rolling_std(pres_level, WINDOW, MINP)
else:
    features["pres_level_mean_5m"] = np.nan
    features["pres_var_t_5m"]      = np.nan

if not cbz_dev.empty:
    features["cbz_dev_mean_5m"]    = _rolling_mean(cbz_dev, WINDOW, MINP)
    features["cbz_dev_std_5m"]     = _rolling_std(cbz_dev, WINDOW, MINP)
else:
    features["cbz_dev_mean_5m"]    = np.nan
    features["cbz_dev_std_5m"]     = np.nan

# ---- Umbrales automáticos y de proceso (sin hard-coding frágil)
# Actividad de línea: hal_speed activo = p10 de valores > 0 (si no hay positivos, fallback a 0)
pos_hal = hal_speed[hal_speed > 0]
speed_active_thresh = _robust_q(pos_hal, 0.10, fallback=0.0)
# para estados, usamos además un umbral "muy bajo" para paro contundente
speed_paro_thresh = 0.5 * speed_active_thresh

# Variabilidad (hi/lo) vía cuantiles
rpm_var_hi  = _robust_q(features["rpm_var_t_5m"].dropna(), 0.75, fallback=np.nan)
rpm_var_lo  = _robust_q(features["rpm_var_t_5m"].dropna(), 0.25, fallback=np.nan)
hal_var_hi  = _robust_q(features["hal_speed_std_5m"].dropna(), 0.75, fallback=np.nan)
hal_var_lo  = _robust_q(features["hal_speed_std_5m"].dropna(), 0.25, fallback=np.nan)
dev_hi      = _robust_q(features["cbz_dev_mean_5m"].dropna(), 0.75, fallback=np.nan)
dev_mid     = _robust_q(features["cbz_dev_mean_5m"].dropna(), 0.50, fallback=np.nan)

# Presión alta: combina mediana empírica y mínimo técnico (300) si existen presiones
if not pres_level.empty:
    pres_med     = _robust_q(features["pres_level_mean_5m"].dropna(), 0.50, fallback=np.nan)
    pres_high_th = np.nanmax([300.0, pres_med])  # prioriza ≥300 si tus transductores trabajan en ese orden
else:
    pres_high_th = np.nan  # sin presión, esta condición se ignora

# % de ceros alto en haladores para PARO
pct_zero_hi = 0.8  # 80% del tiempo y/o haladores en cero dentro de la ventana

# ---- Reglas de estado (orden de prioridad)
# 1) PARO: velocidad casi nula o % de ceros alto
is_paro = (
    (features["hal_speed_mean_5m"] <= speed_paro_thresh) |
    (features["hal_pct_zero_5m"] >= pct_zero_hi)
)

# 2) PRODUCCION: línea activa, variabilidad baja y desviación de cabezal moderada
is_produccion = (
    (features["hal_speed_mean_5m"] >= speed_active_thresh) &
    (
        (pd.isna(rpm_var_lo) | (features["rpm_var_t_5m"] <= rpm_var_lo)) &
        (pd.isna(hal_var_lo) | (features["hal_speed_std_5m"] <= hal_var_lo)) &
        (pd.isna(dev_mid)    | (features["cbz_dev_mean_5m"] <= dev_mid))
    ) &
    (
        # si hay presión, opcionalmente exigimos que no sea absurdamente baja
        (pd.isna(pres_high_th) | (features["pres_level_mean_5m"] >= 0))
    )
) & (~is_paro)

# 3) DESPERDICIO: línea activa + inestabilidad alta y presión alta (si existe) o desviación alta
is_desperdicio = (
    (features["hal_speed_mean_5m"] >= speed_active_thresh) &
    (
        ( (pd.isna(rpm_var_hi) == False) & (features["rpm_var_t_5m"] >= rpm_var_hi) ) |
        ( (pd.isna(hal_var_hi) == False) & (features["hal_speed_std_5m"] >= hal_var_hi) ) |
        ( (pd.isna(dev_hi)     == False) & (features["cbz_dev_mean_5m"] >= dev_hi) )
    ) &
    (
        pd.isna(pres_high_th) | (features["pres_level_mean_5m"] >= pres_high_th)
    )
) & (~is_paro) & (~is_produccion)

# 4) SETUP/AJUSTE: lo que queda cuando no es PARO, ni PRODUCCION, ni DESPERDICIO
labels = pd.Series(index=features.index, dtype='object')
labels.loc[is_paro]        = "PARO"
labels.loc[is_produccion]  = "PRODUCCION"
labels.loc[is_desperdicio] = "DESPERDICIO"
labels.fillna("SETUP", inplace=True)

# ---- Paquete de salida
timeline = features.copy()
timeline["estado"] = labels

# ---- Reportes
counts = timeline["estado"].value_counts(dropna=False).sort_values(ascending=False)
perc = (100 * counts / counts.sum()).round(2)
report = pd.DataFrame({"conteo": counts, "porcentaje": perc})

thresholds = {
    "speed_active_thresh": float(speed_active_thresh) if pd.notna(speed_active_thresh) else None,
    "speed_paro_thresh": float(speed_paro_thresh) if pd.notna(speed_paro_thresh) else None,
    "rpm_var_hi_p75": float(rpm_var_hi) if pd.notna(rpm_var_hi) else None,
    "rpm_var_lo_p25": float(rpm_var_lo) if pd.notna(rpm_var_lo) else None,
    "hal_var_hi_p75": float(hal_var_hi) if pd.notna(hal_var_hi) else None,
    "hal_var_lo_p25": float(hal_var_lo) if pd.notna(hal_var_lo) else None,
    "cbz_dev_hi_p75": float(dev_hi) if pd.notna(dev_hi) else None,
    "cbz_dev_mid_p50": float(dev_mid) if pd.notna(dev_mid) else None,
    "pres_high_thresh": float(pres_high_th) if pd.notna(pres_high_th) else None,
    "pct_zero_hi": float(pct_zero_hi)
}

print("\n=== UMBRALES UTILIZADOS ===")
print(thresholds)
print("\n=== DISTRIBUCIÓN DE ESTADOS (5 min, paso 1 min) ===")
display(report)

# ---- (Opcional) Persistir artefactos
# timeline.to_csv("timeline_estados_5m.csv", index=True)
# features.to_csv("features_5m.csv", index=True)
# pd.Series(thresholds).to_json("umbrales_estado.json")

print("\n✅ BLOQUE 2B completado.")



=== UMBRALES UTILIZADOS ===
{'speed_active_thresh': 60.38333333333333, 'speed_paro_thresh': 30.191666666666666, 'rpm_var_hi_p75': 0.0, 'rpm_var_lo_p25': 0.0, 'hal_var_hi_p75': 0.0, 'hal_var_lo_p25': 0.0, 'cbz_dev_hi_p75': 1.2166666666666615, 'cbz_dev_mid_p50': 1.1433333333333375, 'pres_high_thresh': 3704.88, 'pct_zero_hi': 0.8}

=== DISTRIBUCIÓN DE ESTADOS (5 min, paso 1 min) ===


,conteo,porcentaje
estado,,
PRODUCCION,8716,43.19
SETUP,7828,38.79
DESPERDICIO,3637,18.02



✅ BLOQUE 2B completado. Revisa 'report' y, si quieres, exporta CSV/JSON para el repo.
